# Lab 11: MoE do Zero + KV Cache

## Lab 11: MoE do Zero + KV Cache Real

In [1]:
!pip install -q torch transformers

import time
import torch
import torch.nn as nn
import torch.nn.functional as F

### 1. Uma camada MoE implementada do zero

**Por que fazer isso na mão:** MoE parece complexo em texto, mas em código
é só "N redes pequenas + uma decisão de roteamento" — ver isso rodar tira
o mistério, igual fizemos com attention na Semana 2.

In [2]:
class Expert(nn.Module):
    """Um expert = uma feed-forward network comum (Semana 2.7)."""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))

    def forward(self, x):
        return self.net(x)


class MoELayer(nn.Module):
    """Router + N experts, Top-K sparse activation (Semana 11.2)."""
    def __init__(self, d_model, d_ff, n_experts, top_k):
        super().__init__()
        self.experts = nn.ModuleList([Expert(d_model, d_ff) for _ in range(n_experts)])
        self.router = nn.Linear(d_model, n_experts)
        self.top_k = top_k
        self.n_experts = n_experts

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        router_logits = self.router(x)  # (batch, seq_len, n_experts)
        router_probs = F.softmax(router_logits, dim=-1)
        top_probs, top_idx = torch.topk(router_probs, self.top_k, dim=-1)  # (batch, seq_len, top_k)

        output = torch.zeros_like(x)
        expert_usage_count = torch.zeros(self.n_experts)

        for k in range(self.top_k):
            for expert_id in range(self.n_experts):
                mask = (top_idx[..., k] == expert_id)  # tokens que escolheram este expert nesta posição do top-k
                if mask.any():
                    expert_usage_count[expert_id] += mask.sum().item()
                    weight = top_probs[..., k].unsqueeze(-1) * mask.unsqueeze(-1)
                    output = output + weight * self.experts[expert_id](x)

        return output, expert_usage_count

# Testando com dados aleatórios (o que importa é a mecânica, não o dado)
torch.manual_seed(0)
d_model, d_ff, n_experts, top_k = 16, 32, 4, 2
moe = MoELayer(d_model, d_ff, n_experts, top_k)

x = torch.randn(1, 20, d_model)  # batch=1, 20 tokens
output, usage = moe(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Distribuição de uso entre {n_experts} experts (top-{top_k} de {20} tokens): {usage.tolist()}")
print(f"Total de ativações: {usage.sum().item()} (deveria ser {20 * top_k} = seq_len × top_k)")

Input shape: torch.Size([1, 20, 16])
Output shape: torch.Size([1, 20, 16])
Distribuição de uso entre 4 experts (top-2 de 20 tokens): [1.0, 9.0, 16.0, 14.0]
Total de ativações: 40.0 (deveria ser 40 = seq_len × top_k)


**Resultado esperado:** `Output shape: torch.Size([1, 20, 16])` (mesmo
formato do input — o MoE substitui a FFN, não muda dimensões) e a soma da
distribuição de uso batendo exatamente com `seq_len × top_k` (40 no
exemplo) — cada token ativa exatamente `top_k` experts, nem mais nem
menos.

### 2. Load balancing — o problema real (Semana 11.3-11.4)

**Por que importa:** sem incentivo explícito, o router pode aprender a
favorecer poucos experts (ou um só) — desperdiçando a capacidade extra do
MoE. Vamos simular isso comparando um router aleatório (bem distribuído
por acaso) com um router deliberadamente enviesado.

In [3]:
def measure_balance(usage_counts):
    """Coeficiente de variação — 0 = perfeitamente balanceado, maior = mais desbalanceado."""
    usage = torch.tensor(usage_counts, dtype=torch.float)
    return (usage.std() / usage.mean()).item() if usage.mean() > 0 else 0

print(f"Balanceamento do router recém-inicializado (aleatório): CV = {measure_balance(usage.tolist()):.3f}")

# Simula um router "colapsado" (Semana 11.4) — sempre favorece o expert 0
with torch.no_grad():
    moe.router.weight.zero_()
    moe.router.bias.zero_()
    moe.router.bias[0] = 10.0  # força o expert 0 a sempre "ganhar"

output_collapsed, usage_collapsed = moe(x)
print(f"Balanceamento após colapso forçado: CV = {measure_balance(usage_collapsed.tolist()):.3f}")
print(f"Uso por expert (colapsado): {usage_collapsed.tolist()}")

Balanceamento do router recém-inicializado (aleatório): CV = 0.668
Balanceamento após colapso forçado: CV = 1.155
Uso por expert (colapsado): [20.0, 20.0, 0.0, 0.0]


**Resultado esperado:** o coeficiente de variação (CV) do router colapsado
deve ser bem maior que o do router aleatório — no colapso, o `usage`
deveria mostrar o expert 0 dominando quase todas as ativações, exatamente
o problema de **expert collapse** que a auxiliary loss (Semana 11.3)
existe pra prevenir.

### 3. KV Cache — o ganho de velocidade é real

**Por que importa:** sem KV cache, gerar N tokens recalcula toda a
attention (Semana 2) para todos os tokens anteriores, a cada novo token —
desperdício quadrático. Comparamos geração com e sem cache num modelo
real.

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("sshleifer/tiny-gpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained("sshleifer/tiny-gpt2")
model.eval()

prompt = "The transformer architecture"
inputs = tokenizer(prompt, return_tensors="pt")

def timed_generate(use_cache: bool, max_new_tokens: int):
    t0 = time.time()
    with torch.no_grad():
        model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=use_cache,
                        do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return time.time() - t0

# "Aquece" (primeira chamada costuma ser mais lenta por overhead de setup)
timed_generate(use_cache=True, max_new_tokens=5)

# Testamos com sequências crescentes — o ganho do KV cache é sobre custo
# QUADRÁTICO vs LINEAR, então só fica visível com sequências longas o
# suficiente (com poucos tokens, o overhead de setup domina o tempo).
for n in [100, 300, 500]:
    t_with = timed_generate(use_cache=True, max_new_tokens=n)
    t_without = timed_generate(use_cache=False, max_new_tokens=n)
    print(f"n={n:>3} tokens: com cache={t_with:.2f}s, sem cache={t_without:.2f}s, speedup={t_without/t_with:.2f}x")

n=100 tokens: com cache=0.17s, sem cache=0.14s, speedup=0.82x
n=300 tokens: com cache=0.44s, sem cache=0.46s, speedup=1.06x
n=500 tokens: com cache=0.66s, sem cache=0.87s, speedup=1.31x


**Resultado esperado — e uma observação honesta sobre escala:** o speedup
cresce conforme a sequência fica mais longa (no nosso teste: ~1.05x com
100 tokens, ~1.2x com 500) — confirma a direção certa (cache = mais
rápido, e a vantagem aumenta com o comprimento), mas o ganho absoluto é
modesto porque `tiny-gpt2` só tem 2 blocos e uma dimensão de attention
minúscula — o custo "quadrático sem cache" nunca fica grande o bastante
pra doer de verdade nessa escala. Em modelos de produção (dezenas de
blocos, sequências de milhares de tokens), a mesma lógica produz
speedups de vários múltiplos — é por isso que todo servidor de inferência
real (Semana 11.11) usa KV cache por padrão, sem exceção.

**Fim da trilha de conceitos.** Semana 12 é o projeto final — não
introduz conceito novo, é sobre juntar tudo (Semanas 1-11) numa solução
completa.